# Voronoi Depth Illusion

Each Voronoi cell gets concentric shrinking copies aimed at a random off-center vanishing point,
creating a faux-3D tunneling effect.

In [ ]:
import penpal
from penpal import sampling
import numpy as np

W, H = 8, 10
d = penpal.Drawing(W, H)
pw = penpal.pen_width(0.3)
rng = np.random.default_rng(77)

pts = sampling.poisson_disk(W, H, min_dist=0.7, seed=77)
regions = sampling.voronoi(pts, bounds=(0, 0, W, H))

d.layer('outline', color='black', linewidth=pw)
d.layer('depth', color='#2244AA', linewidth=pw * 0.6)

n_steps = 8  # concentric rings per cell

for region in regions:
    # outline
    d.layer('outline').add(penpal.Paths([region]))

    # centroid + random offset for vanishing point
    centroid = region[:-1].mean(axis=0)  # exclude closing vertex
    # offset up to 40% of the cell's radius from centroid
    radii = np.linalg.norm(region[:-1] - centroid, axis=1)
    max_r = radii.max()
    angle = rng.uniform(0, 2 * np.pi)
    offset_r = rng.uniform(0.1, 0.4) * max_r
    vp = centroid + offset_r * np.array([np.cos(angle), np.sin(angle)])

    # concentric shrinks toward the vanishing point
    for i in range(1, n_steps + 1):
        t = i / (n_steps + 1)
        shrunk = (1 - t) * region + t * vp
        d.layer('depth').add(penpal.Paths([shrunk]))

d

In [ ]:
d.save('output/voronoi_depth.svg')
d.save_layers('output/voronoi_depth')